# 1. Data Collection

**Source:** Google Play Store reviews for 11 Israeli financial apps.

**Volume:** 47,188 reviews. **Period:** 2011 to 2026.

**Segments:** 6 traditional banks, 2 digital banks, 3 credit card issuers.

**Tool:** `google-play-scraper`, wrapped in `scripts/run_scraper.py`.

In [1]:
import pandas as pd
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 200)

import sqlite3
from pathlib import Path

# SQLite connection. The repo ships with the populated DB.
DB_PATH = Path("..") / "data" / "processed" / "reviews.db"
engine = sqlite3.connect(DB_PATH)

## The 11 Apps

Six traditional banks (Hapoalim, Leumi, Discount, Mizrahi-Tefahot, FIBI, Mercantile), two digital banks (Pepper, One Zero), and three credit card issuers (Isracard, Cal, Max).

In [2]:
apps = pd.read_sql("""
SELECT name, name_he, segment 
FROM apps 
ORDER BY segment, name
""", engine)
apps

,name,name_he,segment
0,Cal,כאל,credit_card
1,Isracard,ישראכרט,credit_card
2,Max,מקס,credit_card
3,One Zero,ONE ZERO,digital_bank
4,Pepper,פפר,digital_bank
5,Bank Hapoalim,בנק הפועלים,traditional_bank
6,Bank Leumi,בנק לאומי,traditional_bank
7,Discount Bank,בנק דיסקונט,traditional_bank
8,First International Bank,הבינלאומי,traditional_bank
9,Mercantile Discount,מרכנתיל דיסקונט,traditional_bank


## Scraping & Volume

Each app is fetched via `google-play-scraper`, written to SQLite, and backed up to JSON. The run is resumable and tolerates per-app errors.

In [3]:
stats = pd.read_sql("""
SELECT 
    a.name,
    a.segment,
    COUNT(r.review_id) AS n_reviews,
    MIN(r.review_date) AS earliest,
    MAX(r.review_date) AS latest
FROM apps a
LEFT JOIN reviews r ON a.app_id = r.app_id
GROUP BY a.name, a.segment
ORDER BY n_reviews DESC
""", engine)

stats['earliest'] = pd.to_datetime(stats['earliest']).dt.date
stats['latest'] = pd.to_datetime(stats['latest']).dt.date

print(f"Total: {stats['n_reviews'].sum():,} reviews\n")
stats

Total: 47,188 reviews



,name,segment,n_reviews,earliest,latest
0,Bank Hapoalim,traditional_bank,10476,2011-01-15,2026-05-12
1,Pepper,digital_bank,6915,2017-02-20,2026-05-10
2,Bank Leumi,traditional_bank,5924,2011-02-20,2026-05-10
3,Max,credit_card,4917,2011-06-13,2026-05-12
4,Cal,credit_card,4303,2013-02-06,2026-05-09
5,Mizrahi-Tefahot,traditional_bank,3936,2011-12-11,2026-04-27
6,Isracard,credit_card,3869,2012-05-08,2026-05-12
7,Discount Bank,traditional_bank,3280,2010-12-28,2026-05-12
8,First International Bank,traditional_bank,1956,2016-02-03,2026-05-12
9,One Zero,digital_bank,1033,2022-03-16,2026-05-06


## Sample Reviews

Five random reviews showing what the scraper collected.

In [4]:
sample = pd.read_sql("""
SELECT a.name AS app,
    r.score,
    r.review_date,
    substr(r.content, 1, 180) AS content
FROM reviews r
INNER JOIN apps a ON r.app_id = a.app_id
WHERE r.content IS NOT NULL AND length(r.content) > 30
ORDER BY RANDOM()
LIMIT 5
""", engine)
sample['review_date'] = pd.to_datetime(sample['review_date'])

for _, row in sample.iterrows():
    print(f"\n[{row['app']}] {row['score']}*  {row['review_date'].date()}")
    print(f"  {row['content']}")


[Max] 4*  2020-06-17
  היה נחמד אם אפשר היה להוסיף הערה אישית (טקסט חופשי) ליד כל שורת הוצאה לשם תזכורת בסוף החודש... לגבי מהות ההוצאה (כפי שניתן לעשות בביט).

[Discount Bank] 2*  2015-08-25
  לא מצליחה אפילו ללחוץ על כפתור הכניסה

[Discount Bank] 1*  2012-06-15
  אפליקציה ממש גרועה... קורסת יותר מאשר פועלת..

[Pepper] 5*  2021-05-02
  שירות לקוחות מעולה באתי עם בקשה מאוד מיוחדת והשקיעו מאמצים רבים כדי לעזור לי.

[Bank Hapoalim] 1*  2024-03-08
  קשיים בחיבור, פעמים רבות לא מצליח להכניס את קוד המשתמש-כשמתחיל לכתוב זורק החוצה, כשמחובר לא מצליח לקבל שירות, רק הודעות על חיוב עמלות מיותר, מבטחים שבעוד חודש יזכו מהעמלות, נראה עם
